# Faithfulness scaling — real model run (free Colab GPU)

**What this notebook does:** runs the real black-box truncation/corruption
faithfulness test from `project_overview.md` against a real
DeepSeek-R1-Distill-Qwen model, using this Colab session's own GPU instead
of a paid hosted API.

**Why this exists (instead of a hosted API key):** the daily automated
session that maintains this repo runs in a sandbox that can only reach a
short allowlist of sites (basically just PyPI and GitHub) — it cannot reach
Hugging Face, Together, Fireworks, Groq, *or* OpenRouter directly (all five
were tested and all failed the same way). That's true regardless of which
provider is picked or whether it's free, so an API key wouldn't actually
unblock that sandbox. It also turns out the OpenRouter free tier specifically
(the cheapest hosted option) only allows ~50 requests/day without ever having
spent $10 — and this experiment needs on the order of 1,000+ generation
calls per model size, so free-tier hosted APIs would take weeks per size
even if they were reachable.

Colab's free GPU has no such per-request limit (only a weekly GPU-hour cap,
which is generous relative to what one run needs), needs **no API key, no
provider signup, and no billing information** — just your existing Google
account. That's why this notebook downloads the actual model weights (they're
open-weight, no gated access needed) and runs generation locally instead.

**Before running:**
1. `Runtime` → `Change runtime type` → set **Hardware accelerator = T4 GPU**
   (free tier). If Colab offers you CPU only during peak hours, wait a bit
   and retry — this notebook will not work without a GPU.
2. `Runtime` → `Run all`.
3. When it finishes, download the results file it produces (the last cell
   does this automatically) and drop it into the `results/` folder of your
   local copy of this repo, so the next automated session can pick it up,
   aggregate it, and update the scaling-curve plot.

**Which model size to run:** set `MODEL_SIZE` in the parameters cell below
and run this notebook once per size (1.5B, then 7B, then 14B) — running all
three in one Colab session in sequence works too, it just uses more of your
weekly GPU-hour budget in one sitting. All three fit on the free T4:
1.5B and 7B load in bf16 comfortably; 14B needs the 4-bit option
(`LOAD_IN_4BIT = True`, already the default for 14B below) to fit in the
T4's ~15GB usable VRAM.


## 0. Confirm you actually have a GPU runtime

If this errors or reports no GPU, go back and set
`Runtime → Change runtime type → T4 GPU` before continuing.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 1. Install dependencies

Colab already has `torch`. This adds the pieces this repo's pipeline code
needs that aren't preinstalled.

In [ ]:
%pip install -q transformers accelerate bitsandbytes datasets

## 2. Get this repo's pipeline code

Clones the public repo so this notebook can reuse the exact same
`pipeline/` code (corruption logic, scoring, prompt construction) that the
rest of the project uses — no duplicated logic to keep in sync.

In [ ]:
import os

REPO_URL = "https://github.com/NicolasCroft/faithfulness-scaling.git"
REPO_DIR = "faithfulness-scaling"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
import sys
sys.path.insert(0, ".")

## 3. Parameters — set these, then Run All

`MODEL_SIZE` picks which of the three core sizes to run. `N_PROBLEMS` caps
how many GSM8K problems to pull; `project_overview.md` wants "at least a
few hundred correctly-solved problems per size" for the trend to not be
noise, and not every problem will be solved correctly with the full CoT, so
start higher than the target (a few hundred solved) to leave margin. 300 is
a reasonable starting point for a single Colab session; raise it if GPU-hour
budget allows and lower it if a run is taking too long or Colab disconnects
before finishing.

In [ ]:
MODEL_SIZE = "1.5B"  # one of "1.5B", "7B", "14B"
N_PROBLEMS = 300
SEED = 0

MODEL_IDS = {
    "1.5B": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "7B": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    "14B": "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B",
}
# 14B needs 4-bit quantization to fit a free T4's ~15GB usable VRAM
# (fp16 weights alone would be ~28GB). 1.5B/7B don't need it.
LOAD_IN_4BIT = {"1.5B": False, "7B": False, "14B": True}[MODEL_SIZE]

MODEL_NAME = MODEL_IDS[MODEL_SIZE]
print(f"Running {MODEL_NAME} (4-bit={LOAD_IN_4BIT}) on {N_PROBLEMS} problems, seed={SEED}")

## 4. Load problems and the model

`load_gsm8k` needs real internet access to download the dataset from the
Hugging Face Hub — which this Colab environment has (unlike the daily
automated sandbox), so this is expected to just work here.

In [ ]:
from pipeline.data import load_gsm8k
from pipeline.inference import LocalHFBackend

problems = load_gsm8k(split="test", n=N_PROBLEMS)
print(f"Loaded {len(problems)} problems")

backend = LocalHFBackend(
    model_name=MODEL_NAME,
    load_in_4bit=LOAD_IN_4BIT,
)
# Trigger the actual weight download + load now (rather than lazily on the
# first generate() call) so any failure surfaces here, before the long
# run, not partway through it.
backend._load()
print("Model loaded.")

## 5. Run the truncation/corruption faithfulness test

This is the real experiment: generate CoT + answer for each problem, keep
only the ones solved correctly, apply the standard corruption battery, and
regenerate. This is the slow cell — expect it to take a while (roughly
proportional to model size x number of problems); a progress line prints
every 25 problems so you can gauge pace and decide whether to let it run to
completion or stop early with a smaller `N_PROBLEMS` next time.

In [ ]:
import time

from pipeline.run_experiment import run_for_problem
from pipeline.scoring import FaithfulnessResult, summarize

start = time.time()
traces = []
for i, p in enumerate(problems):
    traces.append(run_for_problem(p, backend, seed=SEED))
    if (i + 1) % 25 == 0:
        elapsed = time.time() - start
        rate = (i + 1) / elapsed
        remaining = (len(problems) - (i + 1)) / rate if rate > 0 else float("nan")
        print(f"{i + 1}/{len(problems)} done, {elapsed/60:.1f} min elapsed, "
              f"~{remaining/60:.1f} min remaining")

solved_traces = [t for t in traces if t.solved_correctly]
print(f"\nSolved correctly with full CoT: {len(solved_traces)}/{len(traces)}")

by_method = {}
for t in solved_traces:
    for corruption, _new_answer, changed in t.corruption_outcomes:
        by_method.setdefault(corruption.method, []).append(changed)

results = [
    FaithfulnessResult(
        model_name=f"deepseek-r1-distill-qwen-{MODEL_SIZE.lower()}",
        corruption_method=method,
        n_problems=len(changes),
        n_answer_changed=sum(changes),
    )
    for method, changes in by_method.items()
]
print(summarize(results))

## 6. Save results

Saves both the per-corruption-method summary (small, easy to eyeball) and
the full per-problem trace (larger, needed for any later re-analysis or
debugging a surprising number) as JSON. **Sanity-check note:** if
`solved_traces` above is well under "a few hundred," the summary numbers
are noisier than `project_overview.md` wants to report as a finding —
that's a judgment call for a human to make when looking at the actual
count, not something to silently accept.

In [ ]:
import json
from datetime import datetime, timezone

os.makedirs("results/raw", exist_ok=True)
timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
summary_path = f"results/raw/{MODEL_SIZE.lower()}_summary_{timestamp}.json"
traces_path = f"results/raw/{MODEL_SIZE.lower()}_traces_{timestamp}.json"

summary_payload = {
    "model_size": MODEL_SIZE,
    "model_name": MODEL_NAME,
    "load_in_4bit": LOAD_IN_4BIT,
    "n_problems_attempted": len(traces),
    "n_solved_correctly": len(solved_traces),
    "seed": SEED,
    "results": [
        {
            "model_name": r.model_name,
            "corruption_method": r.corruption_method,
            "n_problems": r.n_problems,
            "n_answer_changed": r.n_answer_changed,
            "rate": r.rate,
        }
        for r in results
    ],
}
with open(summary_path, "w") as f:
    json.dump(summary_payload, f, indent=2)

traces_payload = [
    {
        "problem_id": t.problem.problem_id,
        "question": t.problem.question,
        "gold_answer": t.problem.answer,
        "model_answer": t.original.final_answer,
        "solved_correctly": t.solved_correctly,
        "corruption_outcomes": [
            {"method": c.method, "new_answer": a, "answer_changed": ch}
            for c, a, ch in t.corruption_outcomes
        ],
    }
    for t in traces
]
with open(traces_path, "w") as f:
    json.dump(traces_payload, f, indent=2)

print(f"Wrote {summary_path}")
print(f"Wrote {traces_path}")

try:
    from google.colab import files
    files.download(summary_path)
    files.download(traces_path)
    print("Triggered browser downloads for both files.")
except ImportError:
    print("Not running in Colab -- files were saved locally, download manually.")

## 7. Next step (do this outside Colab)

Move the two downloaded `.json` files into the `results/raw/` folder of
your local copy of this repo (same relative path they were saved to here).
The next automated session (or you, manually) can then read them,
aggregate across model sizes, and regenerate the scaling-curve plot —
no code changes needed for that step, it's already what
`notebooks/analysis_scaffold.ipynb`'s aggregation logic expects, just
pointed at real files instead of the synthetic demo data.

If you want to run another size next, scroll back up to the parameters
cell, change `MODEL_SIZE`, and use `Runtime → Run all` again (Colab will
skip re-cloning the repo since it already exists in this session).